<a href="https://colab.research.google.com/github/marcohuertas/hugging_face_projects/blob/main/token_classification_value_units_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Token Classification (Version-02)

This notebook contains the code to setup and calibrate a LLM in a token classification task. The tokens to be classified represent the value and unit of a measurement in a sentence.

In this version expands the single unit case ($^∘$C). Add other units that appear not as combinations, e.g. 1 mL, 1 s not as e.g. 1 mL/s

- Dataset: "bowenxian/BioProBench"
- Model checkpoint: "bert-base-cased"

## Prepare data

In [ ]:
import re
from functools import reduce
import numpy as np
from huggingface_hub import notebook_login
from datasets import load_dataset
from datasets import Features, ClassLabel, List
from unicodedata import normalize
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification
from transformers import TrainingArguments, Trainer

### Install packages needed

In [ ]:
# Install these if needed
# !pip install --upgrade --quiet seqeval evaluate
# import evaluate

In [ ]:
notebook_login()

### Prepare NER data
Use the data from dataset bowenxian/BioProBench to create a token classifiction dataset, where the entities are the values and scientific units found in a sentence.
Labels:
- values: B-VALUE, I-VALUE
- units: B-UNIT, I-UNIT



In [ ]:
ds_base = load_dataset("bowenxian/BioProBench", data_files="PQA.json")

def fill_with_answer(example):
  return re.sub(r"_+", example['answer'], example['question'])

ds_base = (
    ds_base['train']
    .filter(lambda u: u['type']=='parameter')
    .filter(lambda u: u['id']!='PQA-PAR-B008888')
    .map(
        lambda u: {'sentence': [re.sub(r"__+", a, q) for a, q in zip(u['answer'], u['question'])]},
        batched=True)
    )

In [ ]:
ds_base

Test with following units: $^\circ$C, L, g (grams)

In [ ]:
### Special symbols used in scientific units
# degree (Celsius) : "\u00B0"
temp_degree = "\u00B0"
# micro (\mu) : regexp = "\u03bc"
micro = "\u03bc"
print(f'{temp_degree}, {micro}')

In [ ]:
def filter_selected_units(sample):
  """
  Check if at least one of the chosen units is in sample
  """
  # Every unit needs to be preceeded either by a number or a white space
  # to avoid picking up words: e.g. if unit is grams (g)
  # 9g (contains unit), dog (does not contain unit)
  space_or_digit = r'[\d]+\s*'

  # --- List of units ---
  units_list = []

  # Degree celsius
  temp_degree = "\u00B0"
  regexp = f'({temp_degree}C)'
  regexp = "".join([space_or_digit, regexp])
  units_list.append(regexp)

  # Liter, including L, uL, mL
  micro = "\u03bc"
  regexp = f'([{micro}mu]*L)'
  regexp = "".join([space_or_digit, regexp])
  # units_list.append(regexp)

  # Grams, including g, mg, kg, ug
  regexp = f'([{micro}mk]*g)'
  regexp = "".join([space_or_digit, regexp])
  units_list.append(regexp)

  # combine
  regexp = "|".join(units_list)

  regexp = normalize('NFD', regexp)
  sample_normalized = normalize('NFD', sample['sentence'])

  unit_search = re.search(regexp, sample_normalized)

  return unit_search is not None
  # return unit_search

In [ ]:
ds = ds_base.filter(filter_selected_units)

In [ ]:
ds

In [ ]:
ds[:10]['sentence']

### Pre-processing
Prepare the data by splitting the entries in 'sentence' into a list of words.

In [ ]:
# Split by white space and parenthesis, then split the '.' from any word, but not from numbers.

def split_word_period(sample):
  regexp = r'([A-Za-z]+[.])'
  o = re.search(regexp, sample)
  if o:
    output = [u for u in re.split('([.])', sample) if u != '']
  else:
    output = [sample]

  return output

def apply_split_word_period(examples):
  output = []
  for sentence in examples['sentence']:
    sentence_split = [u for u in re.split(r'([\s()])', sentence) if u not in [' ','']]
    # output.append(reduce(lambda p,q: p+q, [split_word_period(u) for u in sentence.split()]))
    output.append(reduce(lambda p,q: p+q, [split_word_period(u) for u in sentence_split]))

  return {'words': output}

ds = ds.map(apply_split_word_period, batched=True, batch_size=100)

In [ ]:
def split_comma(examples):
  """
  Split words separated by commas, but don't split numbers.
  Examples:
  1. all,this -> "all", ",", "this
  2. 1,3 -> "1,3"
  """
  regexp = r'([0-9]+,[0-9]+)'

  output = []
  for words in examples['words']:
    new_words_split = []

    for word in words:
      # check that it has a comma
      # and that is not a number (e.g 3,500)
      has_comma = "," in word
      check = re.search(regexp, word)
      if has_comma and check is None:
        word_split = [u for u in re.split("(,)", word) if u != '']
        new_words_split.append(word_split)
      else:
        new_words_split.append([word])

    output.append(reduce(lambda p,q: p+q, new_words_split))

  return {'words': output}

ds = ds.map(split_comma, batched=True, batch_size=100)


In [ ]:
### Check how the tokenizer will perform on the sentences/words
model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
sample = ds[0]['words']
print(sample)
input = tokenizer(sample, is_split_into_words=True)
print(input.tokens())

### Pre-label data

Pre-label each word in 'words' using:
- 'B-COMBINED': if no space between value and unit
- 'B-VALUE': if value followed by unit
- 'B-UNIT': if unit preceeded by value
- 'O': all other cases

The output of the pre-labeling should be something like this:

[50$^∘$C, and, 12.345, $^∘$C]

['B-COMBINED', '0', 'B-VALUE', 'B-UNIT']

This will be split by the tokenizer as

['[CLS]', '50', '##°', '##C', 'and', '12', '.', '345', '°C', '[SEP]']

In [ ]:
def is_numeric(string: str) -> bool:
    # Try to convert the string to a float
    # If the conversion is successful, return True
    try:
        float(string)
        return True
    # If a ValueError is thrown, it means the conversion was not successful
    # This happens when the string contains non-numeric characters
    except ValueError:
        return False

def contains_a_number(sample):
  return bool(re.search(r'\d', sample))

def pre_ner_label(sample, label_all_units=True):
  """
  Pre-label words as one of the following
  - 'B-COMBINED': if no space between value and unit
  - 'B-VALUE': if value followed by unit
  - 'B-UNIT': if unit preceeded by value
  - 'O': all other cases
  """

  names = ['O', 'B-VALUE', 'I-VALUE', 'B-UNIT', 'I-UNIT', 'B-COMBINED', 'I-COMBINED']
  label2id = {k:idx for idx,k in enumerate(names)}

  # List of units
  units_list = []
  # Degree celsius
  temp_degree = "\u00B0"
  regexp = f'({temp_degree}C)'
  units_list.append(regexp)

  # Liter, including L, uL, mL
  micro = "\u03bc"
  regexp = f'([{micro}mu]*L)'
  units_list.append(regexp)

  # Grams, including g, mg, kg, ug
  regexp = f'([{micro}mk]*g)'
  units_list.append(regexp)

  # combine
  regexp = "|".join(units_list)

  regexp = normalize('NFD', regexp)
  ner_labels = ['O']*len(sample)
  ner_tags = [0]*len(sample)

  for idx, word in enumerate(sample):
    has_value = contains_a_number(word)
    contains_a_unit = re.search(regexp, normalize('NFD', word))
    has_unit = contains_a_unit is not None

    if has_value and has_unit:
      label = 'B-COMBINED'
      ner_labels[idx] = label
      ner_tags[idx] = label2id[label]
    elif has_unit and (not has_value):
      pos_start, pos_end = contains_a_unit.span()
      if label_all_units:
        # Option 1
        # Only label UNIT regardless of being preceedd by VALUE
        if pos_start == 0:
          label = 'B-UNIT'
          ner_labels[idx] = label
          ner_tags[idx] = label2id[label]

          cond1 = contains_a_number(sample[idx-1]) # the previous word contains a number
          cond2 = ner_labels[idx-1] != 'B-COMBINED' # the previous label is not combined
          if cond1 and cond2:
            label = 'B-VALUE'
            ner_labels[idx-1] = label
            ner_tags[idx-1] = label2id[label]
      else:
        # Option 2
        # Only label UNIT if it is preceeded by a VALUE
        cond1 = pos_start == 0 # the word is just one of the units
        cond2 = contains_a_number(sample[idx-1]) # the previous word contains a number
        cond3 = ner_labels[idx-1] != 'B-COMBINED' # the previous label is not combined
        if cond1 and cond2 and cond3:
          label = 'B-UNIT'
          ner_labels[idx] = label
          ner_tags[idx] = label2id[label]

          label = 'B-VALUE'
          ner_labels[idx-1] = label
          ner_tags[idx-1] = label2id[label]

  return ner_labels, ner_tags

def apply_pre_ner_label(examples, label_all_units=True):
  samples = examples['words']
  output_labels = []
  output_tags = []
  for sample in samples:
    ner_labels, ner_tags = pre_ner_label(sample, label_all_units=label_all_units)
    output_labels.append(ner_labels)
    output_tags.append(ner_tags)

  return {
      'pre_ner_labels': output_labels,
      'pre_ner_tags': output_tags,
      }

ds = ds.map(apply_pre_ner_label, batched=True, batch_size=100, fn_kwargs={"label_all_units": False})

In [ ]:
sample = ds[50]
print(sample['sentence'])
# print(sample['words'])
# print(sample['pre_ner_labels'])
for w,l in zip(sample['words'], sample['pre_ner_labels']):
  print(f'{w} | {l}')

## Prepare Tokenized Dataset for Training

In [ ]:
# NOTE: This function was provided by HF in the token classification example worked out.
# For detailes see: https://huggingface.co/learn/llm-course/chapter7/2

# To adjust the NER tags use this function
# Here labels==ner_tags
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            # Start of a new word!
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            # Special token
            new_labels.append(-100)
        else:
            # Same word as previous token
            label = labels[word_id]
            # If the label is B-XXX we change it to I-XXX
            # MH: This assumes a specific way of arranging the ner-labels
            # see output of ds['train'].features['ner_tags']
            if label % 2 == 1:
                label += 1
            new_labels.append(label)

    return new_labels

Convert the label B-COMBINED into B-VALUE, I-VALUE and B-UNIT, I-UNIT.
This label occurs when the value and the unit don't have a space in between.
In general, there should be a space, but I didn't want to restrict my analysis to only those cases.

1. Take a 'words' list and tokenize it.
2. Align the pre-ner-tags based on the tokenization result
3. Convert B-COMBINED, I-COMBINED by checking where the unit is located and relabel these tokens as B-UNIT, I-UNIT and B-VALUE, I-VALUE
4. Call the output 'labels'

In [ ]:
def token_in_unit(token, unit):
  word_part = token
  if '##' in token:
    word_part = token.split('##')[-1]

  return normalize('NFD', word_part) in normalize('NFD', unit)

def generate_ner_labels(tokens, labels):

  # List of units
  units_list = []
  # Degree celsius
  temp_degree = "\u00B0"
  regexp = f'({temp_degree}C)'
  units_list.append(regexp)

  # Liter, including L, uL, mL
  micro = "\u03bc"
  regexp = f'([{micro}mu]*L)'
  units_list.append(regexp)

  # Grams, including g, mg, kg, ug
  regexp = f'([{micro}mk]*g)'
  units_list.append(regexp)

  # combine
  units = "|".join(units_list)

  names = ['O', 'B-VALUE', 'I-VALUE', 'B-UNIT', 'I-UNIT']
  label2id = {k:idx for idx,k in enumerate(names)}

  new_labels = labels.copy()
  ner_tag = 'O'
  # for idx, (token, label) in enumerate(zip(inputs.tokens(), labels)):
  for idx, (token, label) in enumerate(zip(tokens, labels)):
    if label>=5:
      if token_in_unit(token, units):
        # If token is part of unit
        if ner_tag in ['B-VALUE', 'I-VALUE']:
          ner_tag = 'B-UNIT'
          ner_label_id = label2id[ner_tag]
          new_labels[idx] = ner_label_id
        elif ner_tag == 'B-UNIT':
          ner_tag = 'I-UNIT'
          ner_label_id = label2id[ner_tag]
          new_labels[idx] = ner_label_id
        elif ner_tag == 'I-UNIT':
          ner_label_id = label2id[ner_tag]
          new_labels[idx] = ner_label_id

      else:
        # If token not part of unit
        if ner_tag == 'O':
          ner_tag = 'B-VALUE'
          ner_label_id = label2id[ner_tag]
          new_labels[idx] = ner_label_id
        elif ner_tag == 'B-VALUE':
          ner_tag = 'I-VALUE'
          ner_label_id = label2id[ner_tag]
          new_labels[idx] = ner_label_id
        elif ner_tag == 'I-VALUE':
          ner_label_id = label2id[ner_tag]
          new_labels[idx] = ner_label_id
    else:
      ner_tag = 'O'

  return new_labels

def tokenize_and_generate_labels(examples):
  tokenized_inputs = tokenizer(
      examples['words'],
      is_split_into_words=True,
      truncation=True
      )
  # Get the preliminary ner tags
  pre_ner_tags = examples['pre_ner_tags']
  # Align labels
  old_labels = [align_labels_with_tokens(tags, tokenized_inputs.word_ids(idx)) for idx, tags in enumerate(pre_ner_tags)]
  # Generate new labels
  new_labels = [generate_ner_labels(tokenized_inputs.tokens(idx), labels) for idx, labels in enumerate(old_labels)]

  tokenized_inputs["temp_labels"] = new_labels

  return tokenized_inputs

In [ ]:
# Testing
# names = ['O', 'B-VALUE', 'I-VALUE', 'B-UNIT', 'I-UNIT']
# label2id = {k:idx for idx,k in enumerate(names)}
# id2label = {idx:k for idx,k in enumerate(names)}

# examples = dstest[100]
# tokenized_inputs = tokenizer(
#       examples['words'],
#       is_split_into_words=True,
#       truncation=False
#       )

# pre_ner_tags = examples['pre_ner_tags']
# old_labels = [align_labels_with_tokens(tags, tokenized_inputs.word_ids(idx)) for idx, tags in enumerate([pre_ner_tags])]
# new_labels = [generate_ner_labels(tokenized_inputs.tokens(idx), labels) for idx, labels in enumerate(old_labels)]
# for t,ol,l in zip(tokenized_inputs.tokens(), old_labels[0], new_labels[0]):
#   if ol>=5:
#     print(f'{t} | {ol} | {l} | {id2label[l]}')

In [ ]:
# model_checkpoint = "bert-base-cased"
# tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
tokenized_dataset = ds.map(
    tokenize_and_generate_labels,
    batched=True,
    # remove_columns=dstest.column_names
    )

tokenized_dataset = tokenized_dataset.add_column(
    name='labels',
    column=tokenized_dataset['temp_labels'],
    feature=List(ClassLabel(names=['O', 'B-VALUE', 'I-VALUE', 'B-UNIT', 'I-UNIT']))
    ).remove_columns(column_names='temp_labels')


In [ ]:
tokenized_dataset.features

There were some cases where the unit of $^\circ$C appeared in unseen formats, like $^\circ$C/s that the code did not handle properly.

Filter data to remove those cases.

In [ ]:
tokenized_dataset_oddcases = tokenized_dataset.filter(lambda u: max(u['labels']) > 4)
tokenized_dataset = tokenized_dataset.filter(lambda u: max(u['labels']) <= 4)

In [ ]:
# idx = 2
# ids = tokenized_dataset_oddcases[idx]['input_ids']
# t = tokenizer.convert_ids_to_tokens(ids)
# l = tokenized_dataset_oddcases[idx]['labels']
# print(tokenizer.decode(ids))
# for tt,ll in zip(t,l):
#   print(f'{tt} | {ll}')

### Prepare a train, validate, test split

The only columns needed for training are:
 - 'input_ids'
 - 'token_type_ids'
 - 'attention_mask'
 - 'labels'

In [ ]:
def split_ds_train_test_val(ds):
  seed = 56
  # Create a train, test split
  ds_train_test = ds.train_test_split(train_size=0.9, seed=seed)

  # Use the previous 'train' to create a new split
  ds_final = ds_train_test['train'].train_test_split(train_size=0.8, seed=seed)

  # Replace the 'test' split with one named 'validation'
  ds_final['validation'] = ds_final.pop('test')

  # Add the 'test' dataset from the first split
  ds_final['test'] = ds_train_test['test']

  return ds_final

In [ ]:
use_these_columns = ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
tokenized_dataset_final = split_ds_train_test_val(tokenized_dataset.select_columns(use_these_columns))

In [ ]:
tokenized_dataset_final

## Train model

Follow steps from the token classification tutorial

In [ ]:
metric = evaluate.load("seqeval")

# Function to compute metrics
# label_names = ds_samples_final['train'].features['ner_tags'].feature.names
label_names = tokenized_dataset_final['train'].features['labels'].feature.names

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [ ]:
# These are required for the token classification task
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
)

args = TrainingArguments(
    "bert-base-cased-sci-units-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset_final["train"],
    eval_dataset=tokenized_dataset_final["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
# Push final model to HF Hub
trainer.push_to_hub()

## Test some examples

In [ ]:
from transformers import pipeline

# Replace this with your own checkpoint
# model_checkpoint = "huggingface-course/bert-finetuned-ner"
# model_checkpoint_local = "bert-finetuned-ner/checkpoint-2046"
model_checkpoint_hub = "m1969m/bert-base-cased-sci-units-ner"

token_classifier_local = pipeline(
    "token-classification", model=model_checkpoint_hub, aggregation_strategy="simple"
)


In [ ]:
def mark_valueunit_pairs(sample, token_classifier):
  """
  Mark values and units in the input string.
  Values are identified as <VALUE>.
  Units are identified as [UNIT].

  EXAMPLE:
  With 4mL of HBSS -> With <4>[mL] of HBSS

  Args:
    sample : string
    token_classifier: instance of pipeline("token-classification")
  """
  tc_output = token_classifier(sample)

  noutputs = len(tc_output)

  if (noutputs % 2) == 0:
    npairs = int(noutputs/2)
    output = []
    idx_last = 0
    for idx_pair in range(0, npairs):
      idx = 2*idx_pair
      # print(idx_pair, idx_last, idx)
      # value
      vstart = tc_output[idx]['start']
      vend = tc_output[idx]['end']
      # print(vstart, vend)
      # unit
      ustart = tc_output[idx+1]['start']
      uend = tc_output[idx+1]['end']
      # print(ustart, uend)

      output.append(sample[idx_last:vstart])
      output.append("<" + sample[vstart:vend] + ">")
      output.append("[" + sample[ustart:uend] + "]")
      idx_last = uend

    output.append(sample[idx_last:])

  return "".join(output)


In [ ]:
sample_str = ds[200]['sentence']
output = mark_valueunit_pairs(sample_str, token_classifier_local)
print(sample_str)
print(output)
